In [1]:

%pip install pandas scikit-learn category_encoders



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import pandas as pd

df = pd.read_csv("student_admission_prediction.csv")

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

display(df.head())


c:\Users\davek\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Dataset loaded successfully
Dataset shape: (500, 4)


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status
0,78,7.56,Yes,0
1,91,7.51,No,0
2,68,8.99,Yes,0
3,54,8.25,Yes,0
4,82,8.51,Yes,0


In [3]:

print("Data types:")
print(df.dtypes)

print("\nCategorical columns:")
print(df.select_dtypes(include=["object", "category"]).columns.tolist())

print("\nWork experience values:")
print(df["Work_Experience"].value_counts(dropna=False))


Data types:
Entrance_Exam_Score      int64
Undergraduate_GPA      float64
Work_Experience            str
Admission_Status         int64
dtype: object

Categorical columns:
['Work_Experience']

Work experience values:
Work_Experience
Yes    262
No     238
Name: count, dtype: int64


C:\Users\davek\AppData\Local\Temp\ipykernel_8448\3858487130.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include=["object", "category"]).columns.tolist())


In [ ]:


df_clean = df.copy()


df_clean["Work_Experience"] = (
    df_clean["Work_Experience"]
    .astype("string")
    .str.strip()
    .str.lower()
)


df_clean["Work_Experience"] = df_clean["Work_Experience"].replace({
    "yes": "Yes",
    "y": "Yes",
    "no": "No",
    "n": "No"
})

print(df_clean["Work_Experience"].value_counts(dropna=False))


Work_Experience
Yes    262
No     238
Name: count, dtype: int64[pyarrow]


In [5]:

df_onehot = pd.get_dummies(
    df_clean,
    columns=["Work_Experience"],
    dtype=int
)

display(df_onehot.head())

print("Encoded columns:")
print(df_onehot.columns.tolist())


,Entrance_Exam_Score,Undergraduate_GPA,Admission_Status,Work_Experience_No,Work_Experience_Yes
0,78,7.56,0,0,1
1,91,7.51,0,1,0
2,68,8.99,0,0,1
3,54,8.25,0,0,1
4,82,8.51,0,0,1


Encoded columns:
['Entrance_Exam_Score', 'Undergraduate_GPA', 'Admission_Status', 'Work_Experience_No', 'Work_Experience_Yes']


In [ ]:

df_frequency = df_clean.copy()


frequency_map = (
    df_frequency["Work_Experience"]
    .value_counts(dropna=False)
    .to_dict()
)


df_frequency["Work_Experience_Frequency"] = (
    df_frequency["Work_Experience"].map(frequency_map)
)

display(
    df_frequency[
        ["Work_Experience", "Work_Experience_Frequency"]
    ].head()
)

print("\nFrequency mapping:")
print(frequency_map)


,Work_Experience,Work_Experience_Frequency
0,Yes,262
1,No,238
2,Yes,262
3,Yes,262
4,Yes,262



Frequency mapping:
{'Yes': 262, 'No': 238}


In [7]:

from sklearn.model_selection import train_test_split

X = df_clean[["Work_Experience"]].copy()
y = df_clean["Admission_Status"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 400
Testing rows: 100


In [ ]:

from category_encoders import TargetEncoder

encoder = TargetEncoder(
    cols=["Work_Experience"],
    smoothing=10
)


X_train_encoded = encoder.fit_transform(X_train, y_train)


X_test_encoded = encoder.transform(X_test)

print("Encoded training data:")
display(X_train_encoded.head())

print("Encoded test data:")
display(X_test_encoded.head())


Encoded training data:


,Work_Experience
340,9.615385e-03
163,9.615385e-03
30,1.694747e-10
166,1.694747e-10
331,9.615385e-03


Encoded test data:


,Work_Experience
131,1.694747e-10
347,9.615385e-03
135,9.615385e-03
346,9.615385e-03
192,1.694747e-10


In [9]:

from sklearn.preprocessing import StandardScaler

numeric_columns = [
    "Entrance_Exam_Score",
    "Undergraduate_GPA"
]

scaler = StandardScaler()

df_scaled = df_clean.copy()

df_scaled[numeric_columns] = scaler.fit_transform(
    df_scaled[numeric_columns]
)

display(df_scaled.head())


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status
0,0.439065,0.066223,Yes,0
1,1.189701,0.032094,No,0
2,-0.138348,1.042295,Yes,0
3,-0.946726,0.537195,Yes,0
4,0.670030,0.714662,Yes,0


In [10]:

df_onehot.to_csv("admissions_onehot_encoded.csv", index=False)
df_frequency.to_csv("admissions_frequency_encoded.csv", index=False)
df_scaled.to_csv("admissions_scaled.csv", index=False)

print("All files saved successfully!")


All files saved successfully!
